# Lesson 03 - Agentic Design Patterns

У овом лекцији истражујемо три основна образца дизајна за изградњу ефикасних AI агената:

1. **Јасна упутства за агента** — Креирање прецизних, улогу дефинишућих упита који усмеравају понашање агента
2. **Структурирани излаз помоћу Pydantic модела** — Осигуравање да агенти враћају предвидљиве, валидационе податке
3. **Агенти са једном одговорношћу** — Дизајнирање фокусираних агената који свако добро обавља једну ствар

Примењиваћемо сваки образац на сценарио **препоруке туристичких дестинација**, постепено градећи систем који може предлагати дестинације, проверавати доступност и руковати логистиком.


## Подешавање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Образац 1: Јасна упутства за агента

Најпрепознатљивији образац је уједно и најједноставнији: писање јасних, детаљних упутстава за вашег агента.

Добра упутства дефинишу:
- **Ко** је агент (персона и тон)
- **Шта** треба да ради (одговорности корак по корак)
- **Како** треба да се понаша (ограничења и стил)

Испод, креирамо туристичког консијерџа агента са изричитим упутствима која обликују сваки одговор који он произведе.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Образац 2: Структурисани излаз са Pydantic моделима

Текст слободног формата је користан за разговор, али системе доле у ланцу занимају структурирани подаци.  
Повезивањем **Pydantic модела** са **алатком функцијом**, можемо:

- Дефинисати прецизну шему за излаз агента  
- Аутоматски валидирати одговоре  
- Поуздано интегрисати резултате агента у логику апликације  

Такође уводимо алатку која враћа детаље о дестинацији како би агент темељио своје препоруке на стварним подацима.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Шаблон 3: Агенти са Једном Одговорношћу

Сложени задаци имају користи од расподеле рада између више фокусираних агената, од којих сваки има једну одговорност:

- **Стручњак за дестинацију** који зна о местима и доступности
- **Планер логистике** који се бави летовима, хотелима и плановима путовања

Ово огледа принцип софтверског инжењерства *раздвајања одговорности* — сваки агент је лакши за тестирање, одржавање и унапређење независно.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Резиме

У овој лекцији применили смо три агенциска дизајнерска обрасца на сценарио препоруке путовања:

| Образац | Кључна Идеја | Корисност |
|---|---|---|
| **Јасна упутства** | Дефинишите персонaж, одговорности и ограничења унапред | Конзистентно, у складу са брендом понашање агента |
| **Структурисани излаз** | Користите Pydantic моделе као формат одговора | Валидациони, машински читљиви резултати |
| **Једна одговорност** | Дајте сваком агенту један фокусиран задатак | Лакше за тестирање, одржавање и комбиновaње |

Ови обрасци се природно састављају — можете комбиновати јасна упутства са структурисаним излазом унутар агента са једном одговорношћу да бисте изградили робусне, производњи спремне системе.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
